# 01 — LangChain Deep Agents Models

**来源:** [LangChain Docs — Deep Agents / Models](https://docs.langchain.com/oss/python/deepagents/models)

LangChain Deep Agents 是一个 **Agent 框架**，支持任意实现了 **tool calling**（工具调用）的 LangChain Chat Model。

本笔记从文档中提取核心知识点，包含：
- 模型命名规范与支持列表
- `init_chat_model` 初始化各类 Provider
- Deep Agents Eval Suite 评测数据解读
- `ProviderProfile` 统一配置管理
- 运行时动态切换模型（Middleware）

## 1. 模型标识格式

格式：`provider:model`

```
google_genai:gemini-3.5-flash
openai:gpt-5.4
anthropic:claude-sonnet-4-6
baseten:zai-org/GLM-5.1
```

| 部分 | 含义 |
|------|------|
| `provider` | LangChain 集成名，即 `init_chat_model` 的 `model_provider` 参数 |
| `model` | Provider 侧模型标识符（可能是简单名或命名空间路径） |

完整 Provider 列表参考： [chat model integrations](https://docs.langchain.com/oss/python/integrations/chat)

## 2. 支持的 Provider 与建议模型

以下模型在 Deep Agents Eval Suite 中通过了测试：

| Provider | 建议模型 |
|----------|----------|
| **Google** | `gemini-3.1-pro-preview`, `gemini-3-flash-preview` |
| **OpenAI** | `gpt-5.4`, `gpt-4o`, `o4-mini`, `gpt-5.2-codex`, `gpt-4o-mini`, `o3` |
| **Anthropic** | `claude-opus-4-6`, `claude-opus-4-5`, `claude-sonnet-4-6`, `claude-sonnet-4`, `claude-sonnet-4-5`, `claude-haiku-4-5`, `claude-opus-4-1` |
| **Open-weight** | `GLM-5`, `Kimi-K2.5`, `MiniMax-M2.5`, `qwen3.5-397B-A17B`, `devstral-2-123B` |

> Open-weight 模型可通过 Baseten、Fireworks、OpenRouter、Ollama 等 Provider 调用。

## 3. 初始化模型：`init_chat_model`

LangChain 提供统一的 `init_chat_model()` 入口，自动识别 Provider。

In [21]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
# ---- OpenAI ----
# os.environ["OPENAI_API_KEY"] = "sk-..."
#openai_model = init_chat_model("gpt-5.4")

# ---- Anthropic ----
# os.environ["ANTHROPIC_API_KEY"] = "sk-..."
#anthropic_model = init_chat_model("claude-sonnet-4-6")

# ---- Google Gemini ----
# os.environ["GOOGLE_API_KEY"] = "..."
#gemini_model = init_chat_model("google_genai:gemini-2.5-flash-lite")

# ---- Azure OpenAI ----
# os.environ["AZURE_OPENAI_API_KEY"] = "..."
# os.environ["AZURE_OPENAI_ENDPOINT"] = "..."
# os.environ["OPENAI_API_VERSION"] = "2025-03-01-preview"
# model = init_chat_model("azure_openai:gpt-5.4", azure_deployment=os.environ["AZURE_OPENAI_DEPLOYMENT_NAME"])

# ---- AWS Bedrock ----
# model = init_chat_model("anthropic.claude-3-5-sonnet-20240620-v1:0", model_provider="bedrock_converse")

# ---- HuggingFace ----
# os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hf_..."
# model = init_chat_model("microsoft/Phi-3-mini-4k-instruct", model_provider="huggingface", temperature=0.7, max_tokens=1024)

# ---- OpenRouter ----
# os.environ["OPENROUTER_API_KEY"] = "sk-..."
# model = init_chat_model("auto", model_provider="openrouter")

# ----
# deepseek ---
# os.environ["DEEPSEEK_API_KEY"] = "sk-..."
deepseek_model = init_chat_model("deepseek-v4-flash")

print("Model objects created successfully (placeholder).")

Model objects created successfully (placeholder).


### 3.1 关键参数

| 参数 | 类型 | 说明 |
|------|------|------|
| `model` | string (required) | 模型名，可带 `provider:` 前缀 |
| `temperature` | number | 输出随机性（0=确定，越高越创意） |
| `max_tokens` | number | 输出最大 Token 数 |
| `timeout` | number | 超时秒数 |
| `max_retries` | number (default: 6) | 最大重试次数（指数退避 + 抖动） |

> **重试策略：** 自动处理网络错误、429（限速）、5xx（服务端错误）。不重试 401、404。
> 不稳定网络下建议 `max_retries=10-15`, `timeout=120s`。

## 4. 模型调用方法

| 方法 | 说明 |
|------|------|
| `invoke(messages)` | 传入消息列表，返回完整响应 |
| `stream(messages)` | 实时流式输出 |
| `batch([...])` | 批量并行请求 |

In [5]:
from langchain.messages import HumanMessage, SystemMessage

# 单条消息
response = deepseek_model.invoke("什么鹦鹉有五彩斑斓的羽毛？")
print(response.content[:200])

# 带对话历史
conversation = [
    SystemMessage("你是一个从英语翻译到中文语的翻译员"),
    HumanMessage("Translate: I love programming."),
]
response = deepseek_model.invoke(conversation)
print(response.content)

许多鹦鹉都拥有五彩斑斓的羽毛，其中最典型且最著名的包括以下几种：

1.  **金刚鹦鹉**：这是最为人熟知的“五彩”代表。例如：
    -   **绯红金刚鹦鹉**：身上有鲜艳的红色、蓝色和黄色。
    -   **蓝黄金刚鹦鹉**：背部蓝色，腹部黄色，颈部绿色。
    -   **红绿金刚鹦鹉**：红色为主，翅膀上有蓝色和绿色。

2.  **彩虹吸蜜鹦鹉**：名副其实，它们的羽毛包含了
我爱编程。


## 5. 在 Deep Agents 中使用模型

两种方式：

1. **字符串形式** — 传入 `provider:model`，自动初始化
2. **Model 实例** — 预先用 `init_chat_model` 配置好再传入

In [6]:
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model

# 方式一：传入字符串（自动初始化）
agent_str = create_deep_agent(model="deepseek-v4-flash")

# 方式二：传入预配置实例
model = init_chat_model(
    model="deepseek-v4-flash",
    thinking_level="medium",
)
agent_instance = create_deep_agent(model=model)

print("Agents 创建成功（占位符）。")

G:\ai_code\langchain_study\.venv\Lib\site-packages\langchain\chat_models\base.py:494: UserWarning: WARNING! thinking_level is not default parameter.
                thinking_level was transferred to model_kwargs.
                Please confirm that thinking_level is what you intended.
  return _init_chat_model_helper(


Agents 创建成功（占位符）。


## 6. ProviderProfile — 统一模型配置

`ProviderProfile` 让你将初始化参数按 **Provider 级别** 或 **模型级别** 注册，避免重复传参。

**规则：**
- Provider 级别键（如 `"openai"`）影响该 Provider 所有模型
- 模型级别键（如 `"openai:gpt-5.4"`）仅影响特定模型，且继承 Provider 级配置

In [7]:
from deepagents import ProviderProfile, register_provider_profile

# Provider 级别：所有 OpenAI 模型使用 temperature=0
register_provider_profile(
    "openai",
    ProviderProfile(init_kwargs={"temperature": 0}),
)

# 模型级别：gpt-5.4 额外设置 reasoning_effort
register_provider_profile(
    "openai:gpt-5.4",
    ProviderProfile(init_kwargs={"reasoning_effort": "medium"}),
)

print("Provider profiles registered.")
print("gpt-5.4 will use temperature=0 (inherited) + reasoning_effort=medium (specific).")

Provider profiles registered.
gpt-5.4 will use temperature=0 (inherited) + reasoning_effort=medium (specific).


## 7. 运行时动态选择模型（Middleware）

通过 **Middleware** 机制，可以在不重建 Agent 的情况下动态切换模型（例如从用户下拉菜单选择）。

核心 API：`@wrap_model_call` 装饰器，拦截每次模型调用。

In [23]:
from langchain_core.messages import AIMessage

# -*- coding: utf-8 -*-
"""
动态模型选择 - 精简版
Qwen（轻量）与 DeepSeek（高性能）自动切换
"""

import os
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.chat_models import init_chat_model
from langchain.tools import tool

# ========== 两个模型 ==========
qwen_model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    base_url=os.getenv("DASHSCOPE_BASE_URL"),
    api_key=os.getenv("DASHSCOPE_API_KEY")
)

deepseek_model = init_chat_model(
    model="deepseek-chat",
    model_provider="deepseek",
    temperature=0.3
)


# ========== 简单工具 ==========
@tool
def get_time() -> str:
    """获取当前时间"""
    from datetime import datetime
    return datetime.now().strftime("%H:%M:%S")


# ========== 动态选择中间件 ==========
@wrap_model_call
def dynamic_selector(request, handler):
    """根据对话长度切换模型：≤3轮用Qwen，>3轮用DeepSeek"""
    messages = request.state.get("messages", [])

    if len(messages) > 3:
        selected = deepseek_model
        print(f"[切换] 第{len(messages)}轮 → DeepSeek")
    else:
        selected = qwen_model
        print(f"[保持] 第{len(messages)}轮 → Qwen")

    return handler(request.override(model=selected))


# ========== 创建 Agent ==========
agent = create_agent(
    model=qwen_model,
    tools=[get_time],
    middleware=[dynamic_selector],
    system_prompt="你是助手，回答简单问题。"
)

# ========== 测试 ==========

message_history = []

questions = [
    "你好",  # 第1轮
    "现在几点",  # 第2轮
    "谢谢",  # 第3轮
    "刚才问过时间，还记得吗",  # 第4轮（触发切换）
    "总结一下我们的对话"  # 第5轮
]

for i, question in enumerate(questions, 1):
    print(f"\n📝 用户 (第{i}轮): {question}")

    # 添加用户消息到历史
    message_history.append(HumanMessage(content=question))

    # 调用 Agent（传入完整历史）
    result = agent.invoke({"messages": message_history})

    # 获取回复
    ai_response = result["messages"][-1].content
    print(f"🤖 助手: {ai_response[:100]}...")

    # 添加助手回复到历史（关键！）
    message_history.append(AIMessage(content=ai_response))

    print("-" * 40)

print(f"\n✅ 测试完成！共进行了 {len(questions)} 轮对话")
print(f"最终消息历史长度: {len(message_history)} 条")


📝 用户 (第1轮): 你好
[保持] 第1轮 → Qwen
🤖 助手: 你好！有什么可以帮助你的吗？...
----------------------------------------

📝 用户 (第2轮): 现在几点
[保持] 第3轮 → Qwen
[切换] 第5轮 → DeepSeek
🤖 助手: 现在是凌晨 **00:28**。...
----------------------------------------

📝 用户 (第3轮): 谢谢
[切换] 第5轮 → DeepSeek
🤖 助手: 不客气！如果还有其他问题，随时问我。😊...
----------------------------------------

📝 用户 (第4轮): 刚才问过时间，还记得吗
[切换] 第7轮 → DeepSeek
🤖 助手: 当然记得！你刚才问过时间，我回答的是 **凌晨 00:28**。😊...
----------------------------------------

📝 用户 (第5轮): 总结一下我们的对话
[切换] 第9轮 → DeepSeek
🤖 助手: 好的，以下是我们的对话总结：

1. **你打招呼**：你说“你好”，我回应并询问有什么可以帮忙。
2. **询问时间**：你问“现在几点”，我查询后回答“凌晨 00:28”。
3. **道谢**：你...
----------------------------------------

✅ 测试完成！共进行了 5 轮对话
最终消息历史长度: 10 条


## 8. Deep Agents Eval Suite 评测结果

LangChain 官方用 6 个维度评测 Agent 模型：

| 维度 | 说明 |
|------|------|
| **File Ops** | 文件读写操作 |
| **Retrieval** | 检索/信息获取 |
| **Tool Use** | 工具调用准确性 |
| **Memory** | 多轮记忆能力 |
| **Conversation** | 对话流畅度/上下文理解 |
| **Summarization** | 文本摘要能力 |

### 关键发现

| 模型 | Overall | 亮点 | 弱项 |
|------|---------|------|------|
| google_genai:gemini-3.5-flash | **82%** | File Ops/Retrieval 满分，Tool Use 90% | Memory 54%, Conv 38% |
| openrouter:z-ai/glm-5.1 | **89%** (最高) | Retrieval 满分, Tool Use 89% | Conv 33% |
| openai:gpt-5.5 | 80% | 全面均衡 | 无突出短板 |
| anthropic:claude-opus-4-7 | 80% | File Ops/Retrieval 满分 | Conv 48% |
| baseten:zai-org/GLM-5 | 77% | File Ops/Retrieval 满分, Tool Use 89% | Memory 44%, Conv 24% |
| fireworks:accounts/fireworks/models/glm-5p1 | 81% | File Ops/Retrieval 满分, Tool Use 87% | Conv 33% |
| fireworks:accounts/fireworks/models/minimax-m2p7 | 79% | File Ops/Retrieval 满分, Tool Use 85% | Conv 43% |
| ollama:minimax-m2.7:cloud | 73% | 全维度偏低 | Memory 38%, Conv 29% |
| openrouter:deepseek/deepseek-v4-flash | 81% | File Ops 满分, Tool Use 90% | Retrieval 80%, Conv 33% |

> **注：** Memory 和 Conversation 是所有模型的共同短板，说明多轮记忆与对话仍是当前模型的瓶颈。
> 评测源码： [github.com/langchain-ai/deepagents/tree/main/libs/evals](https://github.com/langchain-ai/deepagents/tree/main/libs/evals#readme)

## 9. 最佳实践总结

| 场景 | 建议 |
|------|------|
| 默认推荐 | `google_genai:gemini-3.5-flash` 综合 82%，File Ops/Tool Use 强 |
| Tool Use 优先 | `gemini-3.5-flash` (90%) 或 `deepseek-v4-flash` (90%) |
| 最高 Overall | `openrouter:z-ai/glm-5.1` (89%) |
| 文件操作必选 | 选 File Ops=100% 的模型（Gemini 3.5, GLM-5, 等） |
| 对话能力要求高 | `gpt-5.5` (52%) 相对最好 |
| 稳定性 | 用 `ProviderProfile` 集中配置 temperature 等参数 |
| 动态切换 | 用 Middleware 实现运行时模型选择 |
| 重试与超时 | 不稳定网络：`max_retries=10-15`, `timeout=120` |